In [8]:
library(lme4)
library(rstanarm)
library(emmeans)
library(dplyr)

In [9]:
df <- read.csv("processed_data/acceptance_data_for_logistic_and_probit.csv") %>%
  mutate(
    # Convert categorical variables to factors
    tone = as.factor(tone),
    lang = as.factor(lang),
    word = as.factor(word),
    participant = as.factor(participant),
    word_log_freq_z = as.numeric(log(word_freq)),
    
    acceptance = as.integer(as.logical(acceptance)) 
  )

In [10]:
print(contrasts(df$tone))

       tone51
tone21      0
tone51      1


In [4]:
# z-score word_log_freq within each language
df <- df %>%
  group_by(lang) %>%
  mutate(word_log_freq_z = scale(word_log_freq_z)) %>%
  ungroup()

In [4]:
library(dplyr)
library(emmeans)

# 1. Initialize an empty list to store the slope dataframes
results_list <- list()

# 2. Loop through the languages
for (current_lang in c("min", "man")) {
  
  # Get all unique participants for this specific language
  participants <- unique(df$participant[df$lang == current_lang])
  
  for (current_participant in participants) {
    
    # 3. Subset the data for the current iteration
    sub_df <- df %>% 
      filter(lang == current_lang, participant == current_participant)
    
    # Skip if the participant has too little data to fit a model reliably
    if (nrow(sub_df) < 10) next
    
    # 4. Use tryCatch to prevent errors from killing the loop
    tryCatch({
      
      model <- stan_glmer(acceptance ~ level * tone * word_log_freq_z + (1 | word), 
                        data = sub_df, 
                        family = binomial(link = "logit"),
                        adapt_delta = 0.99,
                        refresh = 0)
      
      # Extract the slopes for 'level', grouped by 'tone'
      slopes_emm <- emtrends(model, ~ tone, var = "level")
      
      # Convert the emmGrid object to a standard dataframe
      slopes_df <- as.data.frame(slopes_emm)
      
      # 5. Append our tracking variables
      slopes_df$lang <- current_lang
      slopes_df$participant <- current_participant
      
      # Store this participant's dataframe in our list
      results_list[[length(results_list) + 1]] <- slopes_df
      
      # Print progress to the console
      message("Success: ", current_participant, " in ", current_lang)
      
    }, error = function(e) {
      # If the model fails (e.g., complete separation), print a warning
      message("Failed: ", current_participant, " in ", current_lang, " - ", e$message)
    })
  }
}

# 6. Bind the list of dataframes together into one dataframe
final_slopes_df <- bind_rows(results_list)

# 7. Reorder columns for readability (putting ID info first)
final_slopes_df <- final_slopes_df %>% 
  relocate(participant, lang, tone)

# 8. Save to CSV
write.csv(final_slopes_df, "processed_data/all_participant_slopes_glm.csv", row.names = FALSE)

NOTE: Results may be misleading due to involvement in interactions

Success: P1 in min

NOTE: Results may be misleading due to involvement in interactions

Success: P2 in min

NOTE: Results may be misleading due to involvement in interactions

Success: P7 in min

NOTE: Results may be misleading due to involvement in interactions

Success: P14 in min

NOTE: Results may be misleading due to involvement in interactions

Success: P15 in min

NOTE: Results may be misleading due to involvement in interactions

Success: P16 in min

NOTE: Results may be misleading due to involvement in interactions

Success: P17 in min

NOTE: Results may be misleading due to involvement in interactions

Success: P18 in min

NOTE: Results may be misleading due to involvement in interactions

Success: P19 in min

NOTE: Results may be misleading due to involvement in interactions

Success: P23 in min

NOTE: Results may be misleading due to involvement in interactions

Success: P24 in min

NOTE: Results may be mis

In [ ]:
library(dplyr)
library(emmeans)
library(rstanarm)

# 1. Initialize lists and set up the new text file
results_list <- list()
txt_filename <- "processed_data/logit_frequency_significance.txt"

# Create the text file and write the header row
cat("participant\tlang\tfreq_crosses_zero\tinteraction_crosses_zero\n", file = txt_filename)

# 2. Loop through the languages
for (current_lang in c("min", "man")) {
  
  participants <- unique(df$participant[df$lang == current_lang])
  
  for (current_participant in participants) {
    
    sub_df <- df %>% 
      filter(lang == current_lang, participant == current_participant)
    
    if (nrow(sub_df) < 10) next
    
    tryCatch({
      
      model <- stan_glmer(acceptance ~ level * tone * word_log_freq_z + (1 | word), 
                          data = sub_df, 
                          family = binomial(link = "logit"),
                          adapt_delta = 0.99,
                          refresh = 0)
      
      # ---------------------------------------------------------
      # NEW: Check if the 95% Credible Intervals cross zero
      # ---------------------------------------------------------
      # Get the 95% boundaries for all parameters
      ci_95 <- posterior_interval(model, prob = 0.95)
      
      # Helper function: If lower is negative AND upper is positive, it crosses zero (TRUE)
      check_zero <- function(param) {
        if (param %in% rownames(ci_95)) {
          return(ci_95[param, "2.5%"] < 0 & ci_95[param, "97.5%"] > 0)
        } else {
          return(NA) # In case the parameter is dropped
        }
      }
      
      # Check our two specific effects
      freq_zero <- check_zero("word_log_freq_z")
      interaction_zero <- check_zero("level:word_log_freq_z")
      
      # Append this participant's results directly to the text file
      cat(sprintf("%s\t%s\t%s\t%s\n", 
                  current_participant, current_lang, freq_zero, interaction_zero), 
          file = txt_filename, append = TRUE)
      # ---------------------------------------------------------
      
      # Extract slopes and save to your list as before
      slopes_emm <- emtrends(model, ~ tone, var = "level")
      slopes_df <- as.data.frame(slopes_emm)
      
      slopes_df$lang <- current_lang
      slopes_df$participant <- current_participant
      results_list[[length(results_list) + 1]] <- slopes_df
      
      message("Success: ", current_participant, " in ", current_lang)
      
    }, error = function(e) {
      message("Failed: ", current_participant, " in ", current_lang, " - ", e$message)
    })
  }
}

# Bind and save the slopes CSV
final_slopes_df <- bind_rows(results_list) %>% relocate(participant, lang, tone)
write.csv(final_slopes_df, "processed_data/all_participant_slopes_glm.csv", row.names = FALSE)

NOTE: Results may be misleading due to involvement in interactions

Success: P1 in min

NOTE: Results may be misleading due to involvement in interactions

Success: P2 in min

NOTE: Results may be misleading due to involvement in interactions

Success: P7 in min

NOTE: Results may be misleading due to involvement in interactions

Success: P14 in min

NOTE: Results may be misleading due to involvement in interactions

Success: P15 in min

NOTE: Results may be misleading due to involvement in interactions

Success: P16 in min

NOTE: Results may be misleading due to involvement in interactions

Success: P17 in min

NOTE: Results may be misleading due to involvement in interactions

Success: P18 in min

NOTE: Results may be misleading due to involvement in interactions

Success: P19 in min

NOTE: Results may be misleading due to involvement in interactions

Success: P23 in min

NOTE: Results may be misleading due to involvement in interactions

Success: P24 in min

NOTE: Results may be mis